# LLM Evaluation 4 of 4: Does 80% Confidence Mean 80% Right?

The previous chapter asked what supports an answer. Now imagine the system also labels
that answer "80% confident." What should you expect from that number?

Not that one answer is 80% correct. Across many similar cases given that score, we would
want roughly 80 out of 100 to be correct. Checking that long-run match is **calibration**.

We will compare declared scores with known outcomes, try an adjustment, and ask which
answers we could afford to return. The scores below are a small fixed teaching set,
not confidences collected from the real model in Part 1.

### Follow the decision from score to action

| Question | What we inspect |
| --- | --- |
| Do scores like 0.8 match observed correctness? | Groups of similar scores |
| Would a simple score adjustment help? | Fit on one group, check a different group |
| What errors remain if we answer fewer requests? | Errors among returned answers, alongside how many we return |
| Can confidence override missing factual support? | A separate claim-risk check |

We switch from text examples to repeated labeled outcomes because calibration is about
frequencies, not how persuasive one sentence sounds.
`correct` is 1 for a right answer and 0 for a wrong one.
The first ten rows fit the adjustment; the next ten expose its limitations.

![Conceptual comparison of confidence and observed correctness](images/04-calibration-and-confidence-theory-01.png)

The drawing explains the idea. The actual table below determines what happened in this fixture.

In [ ]:
# Deterministic labeled confidence fixture.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

from evaluation_shared import COURSE_CAVEAT, set_seed

set_seed()
confidence = np.array([.97,.94,.91,.88,.84,.81,.78,.74,.70,.66,.61,.58,.54,.49,.44,.39,.34,.29,.24,.18])
correct = np.array([1,1,0,1,0,1,1,0,1,0,1,0,1,0,0,1,0,0,0,0], dtype=int)
split = 10
print(COURSE_CAVEAT)
print("accuracy=", correct.mean(), "mean_confidence=", confidence.mean())


## Part 1 - Compare similar scores with what actually happened

Put answers with similar confidence into groups, or **bins**.
For each group, compare its mean confidence with the fraction that were correct.
If the group says roughly 90% but only two thirds are right, it is overconfident.

The code keeps that table, then summarizes the gaps as **expected calibration error (ECE)**.
Bigger groups contribute more because they contain more answers.
ECE is not another accuracy score; it measures disagreement between scores and outcomes.

**Predict:** which score region in our twenty rows contributes the most to that disagreement?

In [ ]:
# Reliability bins and expected calibration error.
def reliability_table(scores, labels, bins=5):
    edges = np.linspace(0, 1, bins + 1)
    rows = []
    for index in range(bins):
        lower, upper = edges[index], edges[index + 1]
        mask = (scores >= lower) & (scores < upper if index < bins - 1 else scores <= upper)
        if not mask.any():
            continue
        rows.append({
            "lower": lower, "upper": upper, "count": int(mask.sum()),
            "mean_confidence": float(scores[mask].mean()),
            "accuracy": float(labels[mask].mean()),
        })
    table = pd.DataFrame(rows)
    table["gap"] = table["mean_confidence"] - table["accuracy"]
    table["weight"] = table["count"] / len(scores)
    return table

reliability = reliability_table(confidence, correct)
ece = float((reliability["weight"] * reliability["gap"].abs()).sum())
display(reliability.round(3))
print(f"ECE={ece:.3f}")


In [ ]:
# Static reliability diagram keeps local failures visible.
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="black", label="perfect calibration")
ax.scatter(reliability["mean_confidence"], reliability["accuracy"],
           s=60 + reliability["count"] * 15, label="observed bins")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="mean confidence", ylabel="observed accuracy")
ax.set_title("Reliability before calibration")
ax.legend()
plt.tight_layout()
plt.show()


### Read one bin before reading ECE

Six examples fall in the 0.8-1.0 confidence bin. Their mean score is about **0.892**,
but only **four of six** are correct. That bin is overconfident by about **0.225**.
Its weight is 6/20 because it contains six of the twenty observations.

The weighted gaps add to ECE about **0.166**. Keep the bin table: the one-number summary
does not tell you where the mismatch occurred, and changing the bins changes the summary.

## Part 2 - Can we adjust the score without changing the answer?

Try a single control that softens or sharpens all confidence values. This is
**temperature scaling**. Above 1 softens; below 1 sharpens; 1 leaves the scores alone.

The code turns each binary confidence into a log-odds score, divides by a positive
temperature, then converts it back. It chooses the temperature using only the first ten
declared outcomes. The second ten are used to inspect what happened afterward.

The answer labels never change, and this transformation preserves the ordering of the scores.
It cannot move a wrong answer below a correct one that was previously lower.
**Predict:** must an adjustment that fits one group improve the next group?

In [ ]:
# Convert binary confidence into logits and fit temperature by negative log-likelihood.
eps = 1e-6
logits = np.log(np.clip(confidence, eps, 1 - eps) / np.clip(1 - confidence, eps, 1 - eps))

def sigmoid(values):
    return 1 / (1 + np.exp(-values))

def nll(temperature):
    probabilities = sigmoid(logits[:split] / temperature)
    labels = correct[:split]
    return -np.mean(
        labels * np.log(np.clip(probabilities, eps, 1))
        + (1 - labels) * np.log(np.clip(1 - probabilities, eps, 1))
    )

fitted = minimize_scalar(nll, bounds=(0.2, 8.0), method="bounded").x
calibrated = sigmoid(logits / fitted)
before_test = reliability_table(confidence[split:], correct[split:], bins=4)
after_test = reliability_table(calibrated[split:], correct[split:], bins=4)
ece_before = float((before_test["weight"] * before_test["gap"].abs()).sum())
ece_after = float((after_test["weight"] * after_test["gap"].abs()).sum())
print(f"temperature={fitted:.3f}; test ECE {ece_before:.3f} -> {ece_after:.3f}")


In [ ]:
# Verify rank preservation and compare magnitudes.
comparison = pd.DataFrame({
    "raw": confidence[split:],
    "calibrated": calibrated[split:],
    "correct": correct[split:],
})
display(comparison.round(3))
assert np.array_equal(np.argsort(-confidence), np.argsort(-calibrated))
print("Ranking preserved; temperature cannot repair poor discrimination.")


### The adjustment did not improve this check

The fitted temperature is about **2.894**. On the second half, ECE moves from **0.164**
to about **0.250**: it gets worse.

The first half contains the highest scores and the second half the lowest.
They are not representative random samples of the same score range.
This is a useful failed refinement: fitting a parameter is not proof that it generalizes.
The unchanged ranking also means temperature alone cannot repair a poor order of answers.

## Part 3 - How many answers can we return under an error limit?

Suppose we return only answers above a confidence cutoff. Raising it usually leaves fewer
answers to return. That choice is called **selective prediction**; declining to answer is
**abstention**.

Keep two measurements:

| Reading | Question |
| --- | --- |
| Coverage | What fraction of all requests did we answer? |
| Selective risk | Of the answers we returned, what fraction were wrong? |

Return nothing and coverage is zero. Risk among zero returned answers is **undefined**,
not evidence of a perfect service. The table uses `NaN` for that missing measurement.

Read the actual returned counts in the table below. Its plot leaves undefined-risk
points out rather than drawing an empty returned set as a zero-error success.

In [ ]:
# Empty returned sets have no observed error rate.
test_scores, test_labels = calibrated[split:], correct[split:]
risk_rows = []
for threshold in np.linspace(0.1, 0.9, 17):
    served = test_scores >= threshold
    coverage = served.mean()
    risk = np.nan if not served.any() else 1 - test_labels[served].mean()
    risk_rows.append({
        "threshold": threshold, "coverage": coverage,
        "selective_risk": risk, "served": int(served.sum()),
    })
risk_coverage = pd.DataFrame(risk_rows)
display(risk_coverage.round(3))


In [ ]:
# Plot only defined risk values; empty returned sets are not perfect outcomes.
ordered = risk_coverage.sort_values("coverage")
ax = ordered.plot(x="coverage", y="selective_risk", marker="o", figsize=(7, 4))
ax.set(xlim=(0, 1.05), ylim=(0, 1.05))
ax.set_title("Observed risk versus coverage on the tuning fixture")
plt.tight_layout()
plt.show()

def choose_operating_point(table, max_risk):
    if not 0 <= max_risk <= 1:
        raise ValueError("max_risk must be between zero and one.")
    eligible = table[(table["served"] > 0) & (table["selective_risk"] <= max_risk)]
    if eligible.empty:
        print(f"No positive-coverage operating point meets risk <= {max_risk:.2f}; hold the policy.")
        return None
    return eligible.sort_values("coverage", ascending=False).iloc[0]

MAX_RISK = 0.20
chosen = choose_operating_point(risk_coverage, MAX_RISK)
if chosen is not None:
    print("Candidate tuning point, not a validated release policy:", chosen.to_dict())


### There is no qualifying nonempty policy on this grid

At cutoff **0.50**, three of ten answers are returned and one is wrong: risk is about **0.333**.
At **0.55**, none is returned. That row cannot claim an observed zero-error rate.

`chosen` is therefore `None` at the default 0.20 ceiling, with an explicit hold message.
This concerns the tested grid and fixture, not every possible threshold or future dataset.
We do not loosen the ceiling behind the reader's back.

## Part 4 - A confident unsupported answer still needs review

Our sweep has not earned an approved serving policy. We can still inspect the logic of a
two-check router using separately declared example scores.

1. A high claim-risk score sends the answer to review, regardless of confidence.
2. Otherwise, a low-risk answer above an illustrative confidence threshold can be served.
3. Other rows receive a caveat or a refusal under the example rules.

`ROUTING_DEMO_THRESHOLD` is **0.65 for this illustration**, not a cutoff validated by the
earlier sweep. The names such as `grounded-high` are scenario labels, not the output of a
grounding model. Do not send this demonstration policy to a real gateway as approved evidence.

In [ ]:
# Independent routing illustration; this is not the unavailable sweep-derived policy.
ROUTING_DEMO_THRESHOLD = 0.65
routing_cases = pd.DataFrame([
    {"id": "grounded-high", "confidence": 0.82, "claim_risk": 0.10},
    {"id": "fabricated-high", "confidence": 0.91, "claim_risk": 0.92},
    {"id": "grounded-low", "confidence": 0.38, "claim_risk": 0.12},
    {"id": "ambiguous", "confidence": 0.57, "claim_risk": 0.48},
])

def route(row):
    if row["claim_risk"] >= 0.70:
        return "HOLD_FOR_REVIEW"
    if row["confidence"] >= ROUTING_DEMO_THRESHOLD and row["claim_risk"] < 0.30:
        return "SERVE"
    if row["confidence"] >= 0.40:
        return "SERVE_WITH_CAVEAT"
    return "REFUSE"

routing_cases["route"] = routing_cases.apply(route, axis=1)
display(routing_cases)
assert routing_cases.query("id == 'fabricated-high'").iloc[0]["route"] == "HOLD_FOR_REVIEW"


## Your turn - what would relaxing the error limit buy?

The default `MAX_RISK = 0.20` has no qualifying positive-coverage row on this grid.

1. **Predict:** if we allow risk up to `0.35`, which existing row could qualify?
2. **Change:** alter only `MAX_RISK` in the operating-point cell from `0.20` to `0.35`.
3. **Measure:** rerun that selection and record its coverage, served count and observed risk.
4. **Diagnose:** explain the cost: the newly qualifying row returns three of ten answers,
   with one of those three wrong. A looser limit did not make the model more accurate.
5. **Explain:** say why this tuning exercise needs a further untouched evaluation set before
   a release claim. The separate routing demonstration does not change just because this knob does.

## What did the experiment actually show?

- The declared scores are more confident than their observed outcomes justify.
- The fitted temperature preserved ranking but made test-half ECE **worse**, not better.
- No positive-coverage row on the default sweep met the 20% error limit.
- Returning no answers is not a successful zero-risk operating point.
- The separate router example sends high claim risk to review even at high confidence.

These are useful failures to understand, not numbers to promote as a deployable policy.
The score list is tiny and sorted; its two halves are not a representative calibration study.
We also used the second half to explore cutoffs, so it is decision-tuning data, not a sealed
final test. A real policy needs representative calibration data and a further independent check.

### Continue to the gateway only with justified decisions

[Stage 09](../09-llm-gateway/README.md) decides which approved routes may serve a request.
It must consume versioned quality, safety, support and confidence evidence, not invent it.
This fixture did not establish a positive-coverage policy under the default risk limit;
the demonstration's constants are not approval evidence.
Latency and cost matter only after a route meets the required checks.

<a id="concept-recap"></a>

## Concept recap

Confidence is a score before checking an answer. Correctness is the observed result.
Calibration asks whether similar scores correspond to the right long-run frequencies.
A calibrated 80% group still contains errors.

Compare groups of scores with outcomes before trusting one summary:

```python
gap = mean_confidence - observed_accuracy
ece = (bin_weights * abs(bin_gaps)).sum()
```

These sketches use the roles introduced above. ECE depends on the bins and does not
establish that an individual answer is supported.
Temperature can soften scores while leaving their ordering unchanged; check the result
on new data rather than assuming improvement.

When returning only some answers, report both reach and mistakes:

```python
served = scores >= threshold
coverage = served.mean()
risk = 1 - labels[served].mean() if served.any() else np.nan
```

No returned answers means zero coverage and undefined observed risk, not a perfect policy.

```python
policy = choose_operating_point(risk_coverage, max_risk=0.20)
```

`None` means no tested positive-coverage point qualified. Keep that failure explicit.
A separately chosen demonstration threshold does not repair it.
Check claim support independently, and validate a policy on representative data before
the [gateway](../09-llm-gateway/README.md) is allowed to enforce it.